# Swiss Housing Price Analysis — Walkthrough

Mirrors the pipeline in `src/pipeline.py`:
scrape → clean → LLM enrich → SQLite → pandas → stats → viz.

**Research question.** Which factors (size, rooms, location, features) significantly influence Swiss housing prices, and are those relationships statistically significant?

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

## 1. Scrape

In [ ]:
from src.scraper import SwissHousingScraper, save_raw
scraper = SwissHousingScraper()
raw_rows = scraper.scrape()
save_raw(raw_rows)
print(f'Scraped {len(raw_rows)} listings; cantons seen: {sorted(scraper.seen_cantons)}')
raw_rows[:2]

## 2. Regex cleaning

In [ ]:
from src.cleaning import clean_records
cleaned = clean_records(raw_rows)
cleaned[:2]

## 3. LLM enrichment (OpenAI; regex fallback if no key)

In [ ]:
from src.llm_helper import LLMProcessor, enrich_records
enriched = enrich_records(cleaned, LLMProcessor())
{k: enriched[0][k] for k in ('title', 'llm_balcony', 'llm_parking', 'llm_furnished')}

## 4. Pandas view

In [ ]:
from src.analysis import records_to_dataframe, average_price_by_rooms, filter_expensive, sort_by_price
df = records_to_dataframe(enriched)
df.head()

In [ ]:
average_price_by_rooms(df)

## 5. SQLite + SQL

In [ ]:
from src.database import HousingDatabase
df_for_db = df.copy()
df_for_db['features'] = df_for_db['features'].apply(lambda xs: ','.join(xs) if isinstance(xs, list) else xs)
with HousingDatabase() as db:
    db.save(df_for_db)
    print('avg price by rooms:')
    print(db.avg_price_by_rooms())
    print()
    print('avg price by city:')
    print(db.avg_price_by_location())

## 6. Statistics

In [ ]:
from src.statistics import run_all_tests
for t in run_all_tests(df):
    print(f"{t.name}: stat={t.statistic:.3f} p={t.p_value:.4f} → {t.interpretation}")

## 7. Visualizations

In [ ]:
from src.visualization import scatter_size_price, boxplot_rooms_price, barplot_avg_price_by_city
scatter_size_price(df);

In [ ]:
boxplot_rooms_price(df);

In [ ]:
barplot_avg_price_by_city(df);